# 🎙️ Speech Corpus Pipeline: Student 1 (v14 — Modular Runner Notebook)
Covers **Steps 1–4** of the project brief:
1. Video Ingestion & Metadata extraction (`info.json`, Creative Commons verification).
2. Audio conditioning (16 kHz, Mono, 16-bit PCM WAV + FLAC).
3. AST Acoustic Macro-Classification (`speech`, `music`, `mixed`, `silence`) with non-overlapping regions and pause snapping.
4. Continuous 10ms Short-Time Energy curve & Otsu Adaptive Silence Floor.
5. Cut Engine v2 (Global Shortest-Path DP, Whisper Arabic word verification, edge snapping, 100–200ms margins, 8ms Hann fade, whole music regions).
6. 4 Interactive UI Inspectors with 60 FPS moving cursor + 10-minute ground-truth manual check tool.
7. Measured Quality Control & Safe Dataset Splitting.


In [1]:
# ⚙️ Cell 1: Environment Setup & Modular Package Imports
import sys
from pathlib import Path

# Add repository root to Python path so speech_corpus package is loaded cleanly
REPO_PATH = Path('Audio-Datasets-Creation-System')
if REPO_PATH.exists() and str(REPO_PATH.resolve()) not in sys.path:
    sys.path.insert(0, str(REPO_PATH.resolve()))

import numpy as np
import pandas as pd
import torch
from IPython.display import display, Audio, HTML

# 📦 Import directly from the modular speech_corpus package (v2.2)
from speech_corpus.audio import read_wav, write_wav, extract_audio, extract_audio_hq, DEFAULT_SR
from speech_corpus.downloader import download_video, find_cached_video, ingest_local_file, is_creative_commons
from speech_corpus.ast_model import (
    classify_audio, ast_predict_audio_windows, windows_to_regions,
    refine_region_boundaries, get_ast_engine, CONTENT_LABELS,
    validate_cached_regions, save_regions
)
from speech_corpus.segmentation import (
    compute_energy_db, adaptive_silence_floor,
    compute_unified_voting_grid,
    detect_pauses_silero, detect_pauses_energy,
    detect_speech_activity_overlay, compute_instantaneous_energy_vad_overlay,
    log_human_change
)
from speech_corpus.cut_engine_v2 import (
    cut_four_label_segments, cut_speech_and_mixed_segments,
    Cand, PauseCandidate, plan_unified_dp_cuts, greedy_plan,
    apply_hann_fade
)
from speech_corpus.inspector import (
    plot_spectrogram_with_regions,
    launch_spectrogram_region_editor,
    launch_timeline_pointer_inspector,
    launch_vad_overlay_inspector,
    launch_segment_inspector,
    sample_manual_check_window,
    launch_manual_check,
    summarize_manual_check
)
from speech_corpus.quality_control import (
    validate_segments_integrity,
    generate_qc_markdown_report,
    video_summary,
    build_corpus_manifest,
    split_corpus
)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
dev_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'Host CPU'
print(f"🚀 Environment Ready! • Device: {device.upper()} ({dev_name})")
print('✅ Modular Speech Corpus Package Loaded (Specification Version 2.2 — Unified Four-Label Voting).')


🚀 Environment Ready! • Device: CUDA (NVIDIA GeForce RTX 3050 Laptop GPU)
✅ Modular Speech Corpus Package Loaded (Specification Version 2.2 — Unified Four-Label Voting).


## ⚙️ Pipeline Configuration


In [2]:
# ⚙️ Cell 2: Configuration Parameters
USE_MOCK_DEMO = False          # Set to True for an instant offline smoke test
YOUTUBE_URL = "https://www.youtube.com/watch?v=w-aamvlT5C4&t=2s"  # Video URL (>= 1 hour recommended)
LOCAL_FILE = None              # Optional: path to local .mp4/.wav (e.g. "my_video.mp4")
COOKIES_FILE = None            # Optional: path to cookies.txt if YouTube blocks IP
DATA_ROOT = Path("data")       # Storage directory for all data and segments
ANNOTATOR = "Amr"              # Annotator name for audit trail in changes.csv

SILENCE_THRESHOLD_S = 0.25     # 250ms threshold: minimum pause duration for a 100% safe cut
MAX_HEIGHT = 360               # Resolution cap for fast video download
DATA_ROOT.mkdir(parents=True, exist_ok=True)
print(f"Configured: Annotator='{ANNOTATOR}' | Data Root='{DATA_ROOT.resolve()}'")


Configured: Annotator='Amr' | Data Root='C:\Users\Amr\Downloads\Dsp project\data'


## 📥 Steps 1 & 2: Video Download, Audio Conditioning & Metadata (`info.json`)


In [3]:
# 📥 Cell 3: Audio Ingestion & Metadata
if USE_MOCK_DEMO:
    from speech_corpus.labelling import build_mock_audio
    print("⚡ Generating synthetic multi-class test audio...")
    video_id = "mock_demo_audio"
    video_dir = DATA_ROOT / video_id
    video_dir.mkdir(parents=True, exist_ok=True)
    audio_path = video_dir / "audio.wav"
    audio_samples, gt_df = build_mock_audio(seed=42, sr=DEFAULT_SR)
    write_wav(audio_path, audio_samples, sample_rate=DEFAULT_SR)
    info = {
        "video_id": video_id, "source": "mock", "url": None, "title": "Synthetic Audio Demo",
        "channel": "Synthesizer", "duration_s": round(len(audio_samples) / DEFAULT_SR, 2),
        "license": "Creative Commons Attribution", "is_creative_commons": True,
        "downloaded_at": "2026-11-21"
    }
    (video_dir / "info.json").write_text(json.dumps(info, indent=2, ensure_ascii=False), encoding="utf-8")
elif LOCAL_FILE is not None:
    print(f"📂 Ingesting local file: {LOCAL_FILE}")
    video_id = Path(LOCAL_FILE).stem
    video_dir = DATA_ROOT / video_id
    audio_path, info = ingest_local_file(LOCAL_FILE, video_dir)
    audio_samples, sr = read_wav(audio_path, target_sr=DEFAULT_SR)
else:
    cached = find_cached_video(YOUTUBE_URL, search_dirs=[DATA_ROOT])
    if cached is not None:
        audio_path, info = cached
        video_dir = audio_path.parent
        print(f"⚡ Found cached audio for video [{info.get('video_id')}]: {audio_path}")
        audio_samples, sr = read_wav(audio_path, target_sr=DEFAULT_SR)
    else:
        print(f"📥 Downloading and extracting audio from: {YOUTUBE_URL}")
        from speech_corpus.downloader import extract_video_id
        vid = extract_video_id(YOUTUBE_URL)
        video_dir = DATA_ROOT / vid
        audio_path, info = download_video(YOUTUBE_URL, out_dir=video_dir, cookies_file=COOKIES_FILE)
        audio_samples, sr = read_wav(audio_path, target_sr=DEFAULT_SR)

dur_s = len(audio_samples) / DEFAULT_SR
print(f"🎵 Audio Ready: {dur_s:.1f}s ({dur_s/60:.2f} mins) @ {DEFAULT_SR} Hz | Directory: {video_dir}")
print(f"📄 License: '{info.get('license')}' | Creative Commons: {'✅ YES' if info.get('is_creative_commons') else '⚠️ NO'}")


⚡ Found cached audio for video [w-aamvlT5C4]: data\w-aamvlT5C4\audio.wav
🎵 Audio Ready: 852.8s (14.21 mins) @ 16000 Hz | Directory: data\w-aamvlT5C4
📄 License: 'Standard YouTube License' | Creative Commons: ⚠️ NO


## ⚡ Step 3.0: Millisecond VAD Pauses & Otsu Adaptive Silence Floor
Extracts true acoustic pauses and computes the short-time RMS energy (vectorised cumsum) with an Otsu threshold on the dB histogram. Needed to snap region borders before cutting.


In [4]:
# ⚡ Cell 4: Speech Activity & Adaptive Silence Floor
print("🔍 Extracting VAD speech pauses and vectorised short-time RMS energy...")
vad_timeline_df, pauses_df = detect_speech_activity_overlay(
    audio=audio_samples, sr=DEFAULT_SR,
    silence_threshold_s=SILENCE_THRESHOLD_S,
    min_pause_ms=120, vad_backend="auto"
)

intervals_df, times_arr, energy_db_arr, thresh_db = compute_instantaneous_energy_vad_overlay(
    audio=audio_samples, sr=DEFAULT_SR, vad_timeline_df=vad_timeline_df
)
floor_db, _ = adaptive_silence_floor(energy_db_arr)

sp_dur = vad_timeline_df.loc[vad_timeline_df['state'] == 'SPEECH', 'duration'].sum() if not vad_timeline_df.empty else 0.0
sil_dur = vad_timeline_df.loc[vad_timeline_df['state'] == 'SILENCE', 'duration'].sum() if not vad_timeline_df.empty else 0.0

print(f"• Active Speech Time   : {sp_dur/60:.2f} min ({100*sp_dur/dur_s:.1f}%)")
print(f"• Silence / Pause Time : {sil_dur/60:.2f} min ({100*sil_dur/dur_s:.1f}%)")
print(f"• Adaptive Silence Floor: {floor_db:.1f} dB (Threshold: {thresh_db:.1f} dB)")
print(f"• Pauses Detected      : {len(pauses_df)} pauses (Safe pauses >= 250ms: {sum(pauses_df['is_safe'])})")


🔍 Extracting VAD speech pauses and vectorised short-time RMS energy...
• Active Speech Time   : 9.49 min (66.8%)
• Silence / Pause Time : 4.72 min (33.2%)
• Adaptive Silence Floor: -39.8 dB (Threshold: -30.5 dB)
• Pauses Detected      : 564 pauses (Safe pauses >= 250ms: 389)


## 🧠 Step 3: AST Acoustic Classification & Overlap-Free Region Tiling
Predicts multi-class AudioSet labels (Speech, Music, Silence, Mixed), tiles the audio using window centers (0% overlap bug solved), and snaps region edges to the nearest VAD pause.


In [5]:
# 🧠 Cell 5: AST Classification & Non-Overlapping Region Tiling (with v2.2 Cache Validation)
regions_csv = video_dir / 'regions.csv'
windows_csv = video_dir / 'windows_ast.csv'
FORCE_RECLASSIFY = False

# Cache validation against source ID, duration, and version 2.2
is_valid_cache = (
    not FORCE_RECLASSIFY and
    validate_cached_regions(regions_csv, expected_source_id=video_dir.name, expected_duration=dur_s, expected_version='2.2')
)

if is_valid_cache:
    print(f'⚡ Valid cached regions found (v2.2): {regions_csv}')
    regions_df = pd.read_csv(regions_csv)
    windows_df = pd.read_csv(windows_csv) if windows_csv.exists() else None
else:
    print('🧠 Running Audio Spectrogram Transformer (AST) neural inference...')
    regions_df, windows_df = classify_audio(
        y=audio_samples, sr=DEFAULT_SR, win_s=10.0, hop_s=5.0, batch_size=32,
        pauses_df=pauses_df
    )
    save_regions(regions_df, regions_csv, source_id=video_dir.name, duration=dur_s, version='2.2')
    if windows_df is not None and not windows_df.empty:
        windows_df.to_csv(windows_csv, index=False)
    print(f'Saved: {regions_csv} (v2.2 validated cache) and {windows_csv}')

print(f'Identified {len(regions_df)} non-overlapping acoustic regions across {dur_s:.1f}s:')
display(regions_df.head(10))


⚡ Valid cached regions found (v2.2): data\w-aamvlT5C4\regions.csv
Identified 14 non-overlapping acoustic regions across 852.8s:


,region_id,start,end,duration,label,confidence,urgency,review_priority,status,refined
0,R0001,0.00,42.55,42.55,speech,0.336,0.757,URGENT,unreviewed,True
1,R0002,42.55,52.45,9.90,silence,0.900,0.664,URGENT,unreviewed,True
2,R0003,52.45,76.15,23.70,speech,0.629,0.453,MEDIUM,unreviewed,True
3,R0004,76.15,128.05,51.90,silence,0.900,0.267,LOW,unreviewed,True
4,R0005,128.05,465.95,337.90,speech,0.707,0.329,LOW,unreviewed,True
5,R0006,465.95,477.95,12.00,silence,0.900,0.306,LOW,unreviewed,True
6,R0007,477.95,642.65,164.70,speech,0.740,0.298,LOW,unreviewed,True
7,R0008,642.65,653.50,10.85,mixed,0.526,0.624,MEDIUM,unreviewed,True
8,R0009,653.50,662.05,8.55,music,0.710,0.810,URGENT,unreviewed,True
9,R0010,662.05,677.85,15.80,mixed,0.632,0.453,MEDIUM,unreviewed,True


### 🔍 Interactive Inspector 1: Spectrogram & Macro-Region Editor
Inspect linear spectrogram (0–8 kHz), listen, move start/end boundaries, relabel, split, merge, or delete regions. Every change updates `regions.csv` and is logged to `changes.csv` with annotator name.


In [6]:
# 🔍 Cell 6: [Inspector 1] Spectrogram + Region Editor
launch_spectrogram_region_editor(
    audio=audio_samples,
    regions_df=regions_df,
    video_dir=video_dir,
    sr=DEFAULT_SR,
    user=ANNOTATOR
)


### 📍 Interactive Inspector 2: 60 FPS Real-Time Moving Pointer
Synchronized moving red cursor rendered inside an HTML5 Canvas + Web Audio API player. Scrub through audio and watch the instant voice activity classification update at 60 FPS.


In [7]:
# 📍 Cell 7: [Inspector 2] 60 FPS Moving Pointer Across Waveform & Energy Curve
launch_timeline_pointer_inspector(
    audio=audio_samples,
    pauses_df=pauses_df,
    vad_timeline_df=vad_timeline_df,
    sr=DEFAULT_SR
)


### 🔍 Interactive Inspector 3: No-Speech Pauses & Safe-Zone Visualizer
Inspect individual pauses, measure their millisecond duration, and verify whether they meet the >= 250ms safety clearance threshold.


In [8]:
# 🔍 Cell 8: [Inspector 3] Pauses and Safe Zones Inspector
launch_vad_overlay_inspector(
    audio=audio_samples,
    pauses_df=pauses_df,
    vad_timeline_df=vad_timeline_df,
    sr=DEFAULT_SR
)


## ✂️ Step 4: Cut Engine v2 (Global DP Optimizer, Word Verification & Whole Music)
Executes global shortest-path DP over candidate pauses. Snaps stream edges to pauses, merges short streams (< 5s), checks Arabic stop consonants and word boundaries via faster-whisper (`language="ar"`), preserves music regions as whole (`music_whole`), enforces 100–200ms margins with 8ms Hann fade, and measures greedy ablation side-by-side.


In [9]:
# ✂️ Cell 9: Execute Cut Engine v2 (Unified Four-Label Global DP)
USE_WORD_VERIFICATION = False   # Set to True for faster-whisper Arabic word boundary checks
WORD_MODEL_SIZE = 'small'       # 'small' or 'tiny' for word boundary verification

segments_df = cut_speech_and_mixed_segments(
    audio=audio_samples,
    regions_df=regions_df,
    out_dir=video_dir,
    sr=DEFAULT_SR,
    pauses=pauses_df,
    use_word_verification=USE_WORD_VERIFICATION,
    word_model_size=WORD_MODEL_SIZE,
    run_ablation=True
)

print(f'Generated {len(segments_df)} segments in {video_dir / "segments"}:')
cols_show = [c for c in ['segment_id', 'start', 'end', 'duration', 'primary_label', 'label_composition', 'cut_tier', 'voting_agreement', 'decision_class', 'status'] if c in segments_df.columns]
display(segments_df[cols_show].head(10))


ℹ️ Loaded regions.csv (includes macro-regions and any human edits).
🎯 CUT ENGINE V2 — UNIFIED FOUR-LABEL PIPELINE [PASSED]
• Total segments generated     : 70 (total audio: 14.21 min)
• Primary window (5–15 s)      : 70 / 70 (100.0%)
• Fallback window (3–18 s)     : 0 / 70
• Margin 100–200 ms satisfied  : 100.0%
• Whisper word checks          : verified 0, split risks 0
• Review workload              : 6 / 70 (8.6%) [target 5–15%]
• Mandatory QC gates passed    : 10 / 10
• Output manifest              : data\w-aamvlT5C4\segments.csv
Generated 70 segments in data\w-aamvlT5C4\segments:


,segment_id,start,end,duration,primary_label,label_composition,cut_tier,voting_agreement,decision_class,status
0,1,0.00,8.75,8.75,speech,speech:8.75,HIGH,2/2,STRONG,PASSED
1,2,8.75,20.05,11.30,speech,speech:11.30,HIGH,2/2,STRONG,PASSED
2,3,20.05,31.00,10.95,speech,speech:10.95,HIGH,2/2,STRONG,PASSED
3,4,31.00,43.65,12.65,speech,speech:11.55|silence:1.10,HIGH,2/2,STRONG,PASSED
4,5,43.65,57.15,13.50,silence,silence:8.80|speech:4.70,HIGH,2/2,STRONG,PASSED
5,6,57.15,71.35,14.20,speech,speech:14.20,HIGH,1/2,STRONG,PASSED
6,7,71.35,81.00,9.65,silence,silence:4.85|speech:4.80,HIGH,2/2,STRONG,PASSED
7,8,81.00,95.25,14.25,silence,silence:14.25,HIGH,2/2,STRONG,PASSED
8,9,95.25,104.68,9.43,silence,silence:9.43,HIGH,2/3,STRONG,PASSED
9,10,104.68,113.85,9.17,silence,silence:9.17,HIGH,2/2,STRONG,PASSED


### 🔍 Interactive Inspector 4: Cut Segments & Human Review Queue
Listen to individual cut segments, inspect certainty tiers (HIGH, MEDIUM, LOW), relabel if needed, and mark segments reviewed with audit logging.


In [10]:
# 🔍 Cell 10: [Inspector 4] Segment Review Queue & Audit Logging
launch_segment_inspector(
    audio=audio_samples,
    segments_df=segments_df,
    video_dir=video_dir,
    sr=DEFAULT_SR,
    user=ANNOTATOR
)


AttributeError: 'Series' object has no attribute 'label'

## 📋 Step 5: Measured Quality Control (QC) & Ablation Report
Evaluates real, countable integrity checks (no overlapping segments, duration bounds, margins, audio file validity) and compiles the side-by-side measured ablation table (Greedy Baseline vs. Global DP Optimizer).


In [ ]:
# 📋 Cell 11: Quality Control Integrity (10 Mandatory Gates) & Ablation Report
qc_results = validate_segments_integrity(segments_df, video_dir=video_dir, total_dur=dur_s, sr=DEFAULT_SR)
qc_md = generate_qc_markdown_report(
    segments_df, report_path=video_dir / 'qc_report.md',
    video_dir=video_dir, qc=qc_results
)

passed_cnt = qc_results['summary']['passed_gates']
total_cnt = qc_results['summary']['total_gates']
print(f"QC Status: {qc_results['status']} ({passed_cnt}/{total_cnt} mandatory gates passed)")
for g in qc_results['gates']:
    icon = '✅ PASS' if g['passed'] else '❌ FAIL'
    print(f"  {icon} Gate {g['gate_id']}: {g['name']} — {g['detail']}")
if qc_results['errors']:
    print('❌ Errors:', qc_results['errors'])
if qc_results['warnings']:
    print('⚠️ Warnings:', qc_results['warnings'])

print()
print('QC Report Preview:')
print('-' * 65)
print(qc_md)


QC Status: FAILED (5/6 checks passed)
❌ Errors: ['100-200 ms margins >= 80%: 13.4%']
⚠️ Warnings: ['Review workload 82.2% is outside the 5-15% target.']

QC Report Preview:
-----------------------------------------------------------------
# Audio Labelling QC Report (Student 1)
**Segments:** 366 (366 speech/mixed cuts + 0 music regions) | **Total:** 59.46 min

## 1. Measured Compliance (speech/mixed cuts)
- 5-15 s priority window: `366/366 (100.0%)`
- 15-18 s fallback: `0/366`
- 100-200 ms margins satisfied: `13.4%`
- Cut tiers: HIGH `48`, MEDIUM `17`, LOW `301`
- Human review workload: `82.2%` (target 5-15%)

## 2. Integrity Checks
- ✅ No overlapping segments — 0 overlaps []
- ✅ Planned segments within 5-18 s — 0 out of bounds
- ✅ Priority window 5-15 s >= 90% — 100.0%
- ❌ 100-200 ms margins >= 80% — 13.4%
- ✅ Labels in {speech, music, mixed} — 0 invalid
- ✅ WAV files exist and match durations — 0 missing, 0 duration mismatches
- ⚠️ Review workload 82.2% is outside the 5-15% target.



## 🧪 Step 5.5: 10-Minute Ground-Truth Manual Check (Required by Project Brief)
Samples a random contiguous 10-minute window of segments for manual verification. Listen, tick any errors (wrong label / cut inside word), save, and calculate bad cut calibration rates.


In [ ]:
# 🧪 Cell 12: Ground-Truth Manual Quality Check UI (Full Video & Review Queue)
import ipywidgets as widgets
from IPython.display import display, Audio, HTML
import pandas as pd
import numpy as np
from pathlib import Path

# ---------------------------------------------------------
# 1. Prepare Ground-Truth Full Video Data (manual_check.csv)
# ---------------------------------------------------------
video_dir = Path(video_dir)
video_dir.mkdir(parents=True, exist_ok=True)
manual_csv_path = video_dir / "manual_check.csv"

# Load or initialize all segments from segments_df (Full Video, not just 10 mins)
if "segments_df" in globals() and segments_df is not None and len(segments_df) > 0:
    base_df = segments_df.copy()
else:
    raise ValueError("segments_df is empty or not defined! Please run Cell 9 first.")

# Ensure label column compatibility
if "label" not in base_df.columns and "primary_label" in base_df.columns:
    base_df["label"] = base_df["primary_label"]
elif "primary_label" not in base_df.columns and "label" in base_df.columns:
    base_df["primary_label"] = base_df["label"]

# Determine if a segment is flagged for Human Review (Labelling + Segmentation)
is_flagged_series = (
    (base_df.get("needs_review", 0) == 1) |
    (base_df.get("cut_tier", "HIGH").astype(str).str.upper() == "LOW") |
    (base_df.get("review_priority", "LOW").astype(str).str.upper().isin(["HIGH", "URGENT"])) |
    (base_df.get("confidence", 1.0).astype(float) < 0.60) |
    (base_df.get("cut_type", "").astype(str).str.lower() == "blind")
)
base_df["is_flagged"] = is_flagged_series

# If existing manual_check.csv exists, merge existing annotations
if manual_csv_path.exists():
    try:
        existing_chk = pd.read_csv(manual_csv_path)
        ann_cols = ["segment_id", "checked", "label_wrong", "corrected_label", "cut_inside_word", "cut_bad_boundary", "notes"]
        existing_ann = existing_chk[[c for c in ann_cols if c in existing_chk.columns]]
        if "segment_id" in existing_ann.columns and len(existing_ann) > 0:
            base_df = base_df.merge(existing_ann, on="segment_id", how="left", suffixes=("", "_old"))
    except Exception as e:
        print(f"Notice: loading existing annotations: {e}")

# Ensure annotation columns exist
for col in ["checked", "label_wrong", "cut_inside_word", "cut_bad_boundary"]:
    if col not in base_df.columns:
        base_df[col] = 0
    base_df[col] = base_df[col].fillna(0).astype(int)

for col in ["corrected_label", "notes"]:
    if col not in base_df.columns:
        base_df[col] = ""
    base_df[col] = base_df[col].fillna("").astype(str)

# Save initialized full video check file
base_df.to_csv(manual_csv_path, index=False)
chk_df = pd.read_csv(manual_csv_path)

print(f"🎬 Loaded Full Video: {len(chk_df)} segments ({chk_df['duration'].sum()/60:.1f} min)")
print(f"🔍 Flagged for Human Review (Labelling & Segmentation): {int(chk_df['is_flagged'].sum())}/{len(chk_df)} segments")

# ---------------------------------------------------------
# 2. Build Interactive UI
# ---------------------------------------------------------
LABEL_COLORS = {
    "speech": "#2ca02c",
    "music": "#1f77b4",
    "mixed": "#ff7f0e",
    "silence": "#7f7f7f"
}

# 1. Scope selection: Flagged Only vs Full Video
n_flagged = int(chk_df["is_flagged"].sum())
scope_radio = widgets.RadioButtons(
    options=[
        ("🔍 Flagged for Human Review Only (Labelling & Segmentation)", "flagged"),
        ("🎬 All Segments (Full Video)", "all")
    ],
    value="flagged" if n_flagged > 0 else "all",
    description="Scope:",
    layout=widgets.Layout(width="100%")
)

stats_label = widgets.HTML()
sel = widgets.Dropdown(description="Segment:", layout=widgets.Layout(width="80%"))

# 2. Human Review Controls (Labelling & Segmentation)
c_lbl = widgets.Checkbox(description="❌ Region label is WRONG", value=False, indent=False, layout=widgets.Layout(width="220px"))
corrected_drop = widgets.Dropdown(
    options=["speech", "music", "mixed", "silence"],
    description="Correct To:",
    layout=widgets.Layout(width="240px")
)
c_cut_word = widgets.Checkbox(description="✂️ Cut falls INSIDE a word (Word split risk)", value=False, indent=False, layout=widgets.Layout(width="310px"))
c_cut_bad = widgets.Checkbox(description="⚠️ Unnatural / Bad Cut Boundary", value=False, indent=False, layout=widgets.Layout(width="280px"))
txt_notes = widgets.Text(placeholder="Optional notes or reason for error...", description="Notes:", layout=widgets.Layout(width="90%"))

b_save_next = widgets.Button(description="💾 Save & Next", button_style="success", icon="save", layout=widgets.Layout(width="150px"))
b_mark_good = widgets.Button(description="✔ Mark Correct & Next", button_style="primary", icon="check", layout=widgets.Layout(width="180px"))
b_prev = widgets.Button(description="⏮ Prev", layout=widgets.Layout(width="80px"))
b_next = widgets.Button(description="Next ⏭", layout=widgets.Layout(width="80px"))

info_out = widgets.Output()
audio_out = widgets.Output()

state = {
    "df": chk_df,
    "filtered_indices": []
}

def _sync_filter():
    df = state["df"]
    if scope_radio.value == "flagged":
        sub_idx = df[df["is_flagged"] == True].index.tolist()
        if not sub_idx:
            sub_idx = df.index.tolist()
    else:
        sub_idx = df.index.tolist()
    state["filtered_indices"] = sub_idx

    opts = []
    for i in sub_idx:
        r = df.loc[i]
        sid = r.get("segment_id", i + 1)
        lbl = str(r.get("label", r.get("primary_label", "speech"))).lower()
        tier = str(r.get("cut_tier", "HIGH")).upper()
        st, en, dur = float(r["start"]), float(r["end"]), float(r["duration"])
        is_fl = bool(r.get("is_flagged", False))
        is_done = bool(r.get("checked", 0) == 1)
        
        fl_tag = "⚠️" if is_fl else ""
        done_tag = "✔" if is_done else ""
        text = f"Seg {int(sid):04d} [{lbl.upper()}] [{tier}] {st:.1f}s–{en:.1f}s ({dur:.1f}s) {fl_tag} {done_tag}".strip()
        opts.append((text, i))
        
    sel.options = opts
    n_tot = len(df)
    n_fl = int(df["is_flagged"].sum())
    n_done = int((df["checked"] == 1).sum())
    stats_label.value = (
        f"<div style='font-size:13px; margin:4px 0; color:#555;'>"
        f"<b>Scope Segments:</b> {len(sub_idx)} &nbsp;|&nbsp; "
        f"<b>Flagged for Review (Labelling + Segmentation):</b> {n_fl}/{n_tot} ({100*n_fl/max(1,n_tot):.1f}%) &nbsp;|&nbsp; "
        f"<b>Checked:</b> {n_done}/{n_tot} ({100*n_done/max(1,n_tot):.1f}%)"
        f"</div>"
    )
    if sub_idx:
        if sel.value not in sub_idx:
            sel.value = sub_idx[0]

def show(_=None):
    info_out.clear_output()
    audio_out.clear_output()
    if sel.value is None or sel.value not in state["df"].index:
        return
        
    i = sel.value
    r = state["df"].loc[i]
    
    c_lbl.value = bool(r.get("label_wrong", 0))
    c_cut_word.value = bool(r.get("cut_inside_word", 0))
    c_cut_bad.value = bool(r.get("cut_bad_boundary", 0))
    txt_notes.value = str(r.get("notes", "")) if pd.notnull(r.get("notes")) else ""
    
    lbl = str(r.get("label", r.get("primary_label", "speech"))).lower()
    if pd.notnull(r.get("corrected_label")) and str(r.get("corrected_label")).strip():
        corrected_drop.value = str(r.get("corrected_label")).strip()
    else:
        corrected_drop.value = lbl if lbl in ["speech", "music", "mixed", "silence"] else "speech"

    color = LABEL_COLORS.get(lbl, "#333")
    tier = str(r.get("cut_tier", "HIGH")).upper()
    conf = float(r.get("confidence", 1.0)) if pd.notnull(r.get("confidence")) else 1.0
    cert = float(r.get("cut_certainty", 1.0)) if pd.notnull(r.get("cut_certainty")) else 1.0
    is_fl = bool(r.get("is_flagged", False))
    prio = str(r.get("review_priority", "LOW")).upper()
    rev_reason = str(r.get("review_reason", "none"))
    cut_type = str(r.get("cut_type", "midpoint"))
    v_agree = str(r.get("voting_agreement", "3/3"))
    lead_ms = int(r.get("lead_ms", 150)) if pd.notnull(r.get("lead_ms")) else 150
    trail_ms = int(r.get("trail_ms", 150)) if pd.notnull(r.get("trail_ms")) else 150
    
    status_badge = f"<span style='background:{'#d9534f' if is_fl else '#5cb85c'}; color:white; padding:2px 8px; border-radius:4px; font-weight:bold;'>{'⚠️ REVIEW REQUIRED' if is_fl else '✅ AUTO-ACCEPTED'}</span>"
    checked_badge = "<span style='background:#0275d8; color:white; padding:2px 8px; border-radius:4px;'>✔ CHECKED</span>" if r.get("checked", 0) == 1 else ""

    html_info = f'''
    <div style="border:1px solid #ccc; border-radius:8px; padding:12px; margin:8px 0; background-color:#fcfcfc;">
        <div style="display:flex; justify-content:space-between; align-items:center; margin-bottom:8px;">
            <span style="font-size:16px; font-weight:bold;">Segment #{int(r.get('segment_id', i+1)):04d} &nbsp; ({float(r['start']):.2f}s – {float(r['end']):.2f}s, Dur: {float(r['duration']):.2f}s)</span>
            <div>{status_badge} &nbsp; {checked_badge}</div>
        </div>
        
        <div style="display:grid; grid-template-columns: 1fr 1fr; gap:10px; font-size:13px; line-height:1.6;">
            <div style="background:#fff; border:1px solid #e0e0e0; border-radius:6px; padding:8px 12px;">
                <div style="font-weight:bold; color:#333; margin-bottom:4px; border-bottom:1px solid #eee; padding-bottom:3px;">🏷️ SYSTEM LABELLING</div>
                <div><b>Predicted Label:</b> <span style="background:{color}; color:white; padding:2px 7px; border-radius:3px; font-weight:bold;">{lbl.upper()}</span></div>
                <div><b>Label Confidence:</b> {conf:.1%} &nbsp;|&nbsp; <b>Composition:</b> {r.get('label_composition', lbl)}</div>
                <div><b>Source Regions:</b> {r.get('source_region_ids', 'N/A')}</div>
            </div>
            
            <div style="background:#fff; border:1px solid #e0e0e0; border-radius:6px; padding:8px 12px;">
                <div style="font-weight:bold; color:#333; margin-bottom:4px; border-bottom:1px solid #eee; padding-bottom:3px;">✂️ SYSTEM SEGMENTATION</div>
                <div><b>Certainty Tier:</b> <span style="font-weight:bold;">{tier}</span> (Score: {cert:.2f}) &nbsp;|&nbsp; <b>Priority:</b> {prio}</div>
                <div><b>Cut Type:</b> {cut_type} &nbsp;|&nbsp; <b>Voting Agreement:</b> {v_agree}</div>
                <div><b>Silence Margins:</b> Lead: {lead_ms}ms, Trail: {trail_ms}ms &nbsp;|&nbsp; <b>Reason:</b> <code style="color:#c7254e;">{rev_reason}</code></div>
            </div>
        </div>
    </div>
    '''
    with info_out:
        display(HTML(html_info))
    with audio_out:
        sr_rate = DEFAULT_SR if "DEFAULT_SR" in globals() else 16000
        st_samp = int(float(r["start"]) * sr_rate)
        en_samp = int(float(r["end"]) * sr_rate)
        if "audio_samples" in globals() and audio_samples is not None:
            display(Audio(audio_samples[st_samp:en_samp], rate=sr_rate))

def _save_feedback(mark_correct=False):
    i = sel.value
    if i is None or i not in state["df"].index:
        return
        
    if mark_correct:
        lbl_w = 0
        cut_w = 0
        cut_b = 0
        corr_lbl = ""
    else:
        lbl_w = int(c_lbl.value)
        cut_w = int(c_cut_word.value)
        cut_b = int(c_cut_bad.value)
        corr_lbl = corrected_drop.value if lbl_w else ""
        
    notes_val = txt_notes.value.strip()
    state["df"].loc[i, ["checked", "label_wrong", "cut_inside_word", "cut_bad_boundary", "corrected_label", "notes"]] = [
        1, lbl_w, cut_w, cut_b, corr_lbl, notes_val
    ]
    state["df"].to_csv(manual_csv_path, index=False)
    
    cur_list = state["filtered_indices"]
    if i in cur_list:
        pos = cur_list.index(i)
        if pos + 1 < len(cur_list):
            sel.value = cur_list[pos + 1]
        else:
            _sync_filter()
    else:
        _sync_filter()

def on_prev(_):
    cur_list = state["filtered_indices"]
    if sel.value in cur_list:
        pos = cur_list.index(sel.value)
        if pos > 0:
            sel.value = cur_list[pos - 1]

def on_next(_):
    cur_list = state["filtered_indices"]
    if sel.value in cur_list:
        pos = cur_list.index(sel.value)
        if pos + 1 < len(cur_list):
            sel.value = cur_list[pos + 1]

scope_radio.observe(lambda _: _sync_filter(), names="value")
sel.observe(show, names="value")
b_save_next.on_click(lambda _: _save_feedback(mark_correct=False))
b_mark_good.on_click(lambda _: _save_feedback(mark_correct=True))
b_prev.on_click(on_prev)
b_next.on_click(on_next)

ui = widgets.VBox([
    widgets.HTML("<h3 style='margin-bottom:4px;'>🧪 Ground-Truth Manual Quality Check (Full Video & Review Queue)</h3>"),
    scope_radio,
    stats_label,
    widgets.HBox([b_prev, sel, b_next]),
    info_out,
    audio_out,
    widgets.HTML("<div style='font-weight:bold; margin-top:8px; margin-bottom:4px;'>Human Reviewer Annotations:</div>"),
    widgets.HBox([c_lbl, corrected_drop]),
    widgets.HBox([c_cut_word, c_cut_bad]),
    txt_notes,
    widgets.HBox([b_save_next, b_mark_good], layout=widgets.Layout(margin="8px 0"))
])

_sync_filter()
show()
display(ui)


Manual-check sample: 60 segments, 10.0 min -> manual_check.csv


In [ ]:
# 📊 Cell 13: Summarize Manual Check Results
# Run this cell after you have finished ticking the segments in the checklist above
manual_stats = summarize_manual_check(video_dir)


Checked 0/60 segments (0.0 min)


## 📂 Step 6: Summary Table, Corpus Manifest & Zero-Leakage Dataset Splitting
Generates `video_summary.csv` with durations of speech, music, mixed, and silence. Compiles all videos' segments into `corpus_manifest.csv` and performs safe cross-video (or time-block) dataset splitting (70% Train, 15% Val, 15% Test).


In [ ]:
# 📂 Cell 14: Summary, Manifest & Zero-Leakage Splitting
summary_df = video_summary(
    regions_df=pd.read_csv(video_dir / 'regions.csv'),
    segments_df=segments_df,
    info=info,
    video_dir=video_dir
)
print('📊 Video Duration & Labelling Summary:')
display(summary_df)

# Global corpus manifest across all processed videos
manifest_df = build_corpus_manifest(DATA_ROOT, DATA_ROOT / 'corpus_manifest.csv')
print(f"📄 Corpus Manifest: {len(manifest_df)} total segments from {manifest_df['video_id'].nunique()} video(s)")

# Dataset splitting (strictly source-level, zero leakage)
splits = split_corpus(manifest_df, out_dir=DATA_ROOT / 'splits', ratios=(0.70, 0.15, 0.15))
if splits.get('status') == 'SUCCESS':
    print(f"✅ Splits Saved in {DATA_ROOT / 'splits'}: Train={len(splits['splits']['train'])}, Val={len(splits['splits']['val'])}, Test={len(splits['splits']['test'])}")
else:
    print(f"ℹ️ Split Result: {splits.get('message', 'Infeasible (fewer than 3 sources)')}")


📊 Video Duration & Labelling Summary:


,video,duration_min,speech_min,music_min,mixed_min,silence_min,segments,cut_segments,mean_seg_s,review_pct,license,url
0,إيران بين شبح الاحتجاجات الداخلية واستنفار حام...,59.84,56.81,0,3.02,0,366,366,9.75,82.2,Standard YouTube License,https://www.youtube.com/watch?v=7fIN_zj4Yno


📄 Corpus Manifest: 428 total segments from 1 video(s)
⚠️ Fewer than 3 videos: using time-block splits (possible speaker overlap between splits).
✅ Splits Saved in data\splits: Train=256, Val=55, Test=55


## 💾 Step 7: Packaging & Downstream Handoff
Creates a unified zip archive containing `segments.csv`, audio WAV files, metadata, QC reports, and splits, ready for handoff to Student 2 (Denoising / Noise Filtering) and Student 3 (Web App / UI).


In [ ]:
# 💾 Cell 15: Create Handoff Zip Archive
import zipfile

zip_filename = f"{video_dir.name}_student1_handoff.zip"
with zipfile.ZipFile(zip_filename, "w", zipfile.ZIP_DEFLATED) as z:
    for f in video_dir.rglob("*"):
        if f.is_file() and not f.name.endswith(".zip"):
            z.write(f, arcname=f.relative_to(video_dir.parent))

print(f"🎉 Pipeline Complete! Handoff archive created: {Path(zip_filename).resolve()} ({Path(zip_filename).stat().st_size / (1024*1024):.2f} MB)")
print("🚀 Ready for handoff to Student 2 and Student 3!")


🎉 Pipeline Complete! Handoff archive created: C:\Users\Amr\Downloads\Dsp project\7fIN_zj4Yno_student1_handoff.zip (638.02 MB)
🚀 Ready for handoff to Student 2 and Student 3!
